In [ ]:


import pandas as pd
import numpy as np
import re
from collections import Counter

In [ ]:
import pandas as pd

df = pd.read_parquet(
    "https://storage.googleapis.com/msca-bdp-data-open/news_final_project/news_final_project.parquet",
    engine="pyarrow"
)

print(df.shape)
df.head()

(199989, 5)


,url,date,language,title,text
0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod..."
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,\n\nThis AI video of gymnastics might be the f...
2,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...","\n\nIf using AI feels like a chore, try this -..."
3,https://citylife.capetown/gl/uncategorized/the...,2023-11-10,en,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation M...
4,https://citylife.capetown/kk/uncategorized/mic...,2023-11-19,en,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers ...


In [ ]:
print(df.columns.tolist())
df.info()

['url', 'date', 'language', 'title', 'text']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199989 entries, 0 to 199988
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   url       199989 non-null  object
 1   date      199989 non-null  object
 2   language  199989 non-null  object
 3   title     199989 non-null  object
 4   text      199989 non-null  object
dtypes: object(5)
memory usage: 7.6+ MB


In [ ]:
df.sample(5, random_state=42)

,url,date,language,title,text
129081,https://en.cedarnews.net/83862/task/researcher...,2024-01-18,en,Researchers have developed an AI tool that can...,Researchers have developed an AI tool that c...
175075,https://www.canindia.com/data-science-and-mach...,2022-10-10,en,'Data Science' and 'Machine Learning' at IIT M...,\n\n'Data Science' and 'Machine Learning' at ...
182259,https://www.kxlf.com/business/company-news/elo...,2024-06-11,en,Elon Musk threatens to ban workers from using ...,\n\nElon Musk threatens to ban workers from us...
141850,https://www.livemint.com/companies/news/big-te...,2026-01-16,en,Big tech to soon pay for power costs for AI da...,Big tech to soon pay for power costs for AI da...
67774,https://citylife.capetown/nl/uncategorized/how...,2023-12-09,en,How close is AI to replacing humans?,How close is AI to replacing humans? \n \...


In [ ]:
URL_COL = "url"
DATE_COL = "date"
LANG_COL = "language"
TITLE_COL = "title"
TEXT_COL = "text"

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100
}).sort_values("missing_count", ascending=False)

missing

,missing_count,missing_pct
url,0,0.0
date,0,0.0
language,0,0.0
title,0,0.0
text,0,0.0


In [ ]:
df[LANG_COL].value_counts(dropna=False).head(20)

,count
language,
en,199989


In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate URLs:", df[URL_COL].duplicated().sum())
print("Duplicate titles:", df[TITLE_COL].duplicated().sum())

Exact duplicate rows: 0
Duplicate URLs: 0
Duplicate titles: 35684


In [ ]:
df_clean = df.copy()

df_clean[DATE_COL] = pd.to_datetime(df_clean[DATE_COL], errors="coerce")

print("Invalid dates:", df_clean[DATE_COL].isna().sum())
print("Min date:", df_clean[DATE_COL].min())
print("Max date:", df_clean[DATE_COL].max())
print(df_clean[DATE_COL].dt.year.value_counts().sort_index())

Invalid dates: 0
Min date: 2022-01-01 00:00:00
Max date: 2026-02-10 00:00:00
date
2022    18361
2023    71383
2024    49789
2025    54345
2026     6111
Name: count, dtype: int64


In [ ]:
def clean_text_basic(text):
    if pd.isna(text):
        return np.nan

    text = str(text)
    text = text.replace("\\n", " ").replace("\\t", " ").replace("\\r", " ")
    text = text.replace("\n", " ").replace("\t", " ").replace("\r", " ")
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_clean[TITLE_COL] = df_clean[TITLE_COL].apply(clean_text_basic)
df_clean[TEXT_COL] = df_clean[TEXT_COL].apply(clean_text_basic)

In [ ]:
df_clean["full_text"] = (
    df_clean[TITLE_COL].fillna("") + " " + df_clean[TEXT_COL].fillna("")
).str.strip()

df_clean["title_len"] = df_clean[TITLE_COL].str.len()
df_clean["text_len"] = df_clean[TEXT_COL].str.len()
df_clean["full_text_len"] = df_clean["full_text"].str.len()

df_clean[["title_len", "text_len", "full_text_len"]].describe()

,title_len,text_len,full_text_len
count,199989.000000,199989.000000,199989.000000
mean,86.182335,8390.490187,8477.672522
std,37.528661,6210.915715,6216.217223
min,6.000000,20.000000,41.000000
25%,66.000000,4943.000000,5027.000000
50%,81.000000,7157.000000,7246.000000
75%,101.000000,10445.000000,10533.000000
max,8250.000000,230322.000000,230423.000000


In [ ]:
before = len(df_clean)

df_clean = df_clean[
    df_clean["full_text"].notna() &
    (df_clean["full_text_len"] > 100)
].copy()

after = len(df_clean)

print("Removed short/empty rows:", before - after)
print("Remaining rows:", after)

Removed short/empty rows: 108
Remaining rows: 199881


In [ ]:
before = len(df_clean)

df_clean = df_clean.drop_duplicates(subset=[URL_COL])
df_clean = df_clean.drop_duplicates(subset=[TITLE_COL, TEXT_COL])

after = len(df_clean)

print("Removed duplicates:", before - after)
print("Remaining rows:", after)

Removed duplicates: 501
Remaining rows: 199380


In [ ]:
df_clean.sort_values("full_text_len")[[DATE_COL, TITLE_COL, TEXT_COL, "full_text_len"]].head(10)

,date,title,text,full_text_len
175345,2023-05-16,'Artificial Intelligence' Curse or Boon? | San...,'Artificial Intelligence' Curse or Boon? | San...,101
169664,2026-01-24,Sayuri's AI Boyfriend and Rental Date Reveal T...,Sayuri's AI Boyfriend and Rental Date Reveal T...,101
196232,2025-12-25,"Shinsegae Chairman Chung, Vance Discuss AI Exp...","Shinsegae Chairman Chung, Vance Discuss AI Exp...",101
132467,2025-09-24,U.S. ICE deploys AI tech in deportation operat...,U.S. ICE deploys AI tech in deportation operat...,101
29378,2025-10-06,WMO Bridges Weather Gap with AI Forecasting Sy...,WMO Bridges Weather Gap with AI Forecasting Sy...,101
171145,2026-02-09,NVIDIA Halts OpenAI Investment Amid CEO Skepti...,NVIDIA Halts OpenAI Investment Amid CEO Skepti...,101
7602,2026-01-17,Google's AI Overview Offers Inaccurate Health ...,Google's AI Overview Offers Inaccurate Health ...,101
182383,2024-01-29,"""le petit negre"" - Claude Debussy - piano tuto...","""le petit negre"" - Claude Debussy - piano tuto...",101
33713,2025-12-14,Krafton Announces Fourth AI Fellowship Recruit...,Krafton Announces Fourth AI Fellowship Recruit...,101
122617,2025-11-01,Japan Demands OpenAI Halt Japanese IP Infringe...,Japan Demands OpenAI Halt Japanese IP Infringe...,101


In [ ]:
df_clean.sort_values("full_text_len", ascending=False)[[DATE_COL, TITLE_COL, TEXT_COL, "full_text_len"]].head(10)

,date,title,text,full_text_len
36250,2025-08-28,"Eliezer Yudkowsky - Why AI Will Kill Us, Align...","Eliezer Yudkowsky - Why AI Will Kill Us, Align...",230423
14746,2024-01-13,Tackling COVID-19 Through Responsible AI Innov...,Tackling COVID-19 Through Responsible AI Innov...,189316
10580,2023-01-11,Spielberg trionfa ai Golden Globe con The Fabe...,Spielberg trionfa ai Golden Globe con The Fabe...,183261
71875,2024-01-17,Mayor's Charity Gala supports philanthropic pr...,Mayor's Charity Gala supports philanthropic pr...,152550
57126,2025-03-04,Can his Majesty Official Opposition request th...,Can his Majesty Official Opposition request th...,149651
2812,2025-08-28,Holden Karnofsky - Transformative AI & Most Im...,Holden Karnofsky - Transformative AI & Most Im...,135104
23492,2025-11-19,Transcript: House Hearing on the Risks and Ben...,Transcript: House Hearing on the Risks and Ben...,132957
51230,2024-08-28,Image Recognition Market | The Rise Of AI In R...,Image Recognition Market | The Rise Of AI In R...,118935
89893,2024-12-05,Machine Learning in Medical Imaging Market Key...,Machine Learning in Medical Imaging Market Key...,115858
96473,2022-06-16,Daniel Bard shares baseball with his children,Daniel Bard shares baseball with his children ...,115767


In [ ]:
df_clean.to_parquet("news_cleaned_notebook1.parquet", index=False)
df_clean.sample(min(5000, len(df_clean)), random_state=42).to_parquet("news_sample_5k.parquet", index=False)

print("Saved cleaned files.")

Saved cleaned files.


In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_parquet("news_cleaned_notebook1.parquet", engine="pyarrow")
print(df.shape)
df.head()

(199380, 9)


,url,date,language,title,text,full_text,title_len,text_len,full_text_len
0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...",77,3500,3578
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,78,4820,4899
2,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi...",54,5123,5178
3,https://citylife.capetown/gl/uncategorized/the...,2023-11-10,en,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...,102,4017,4120
4,https://citylife.capetown/kk/uncategorized/mic...,2023-11-19,en,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,65,4290,4356


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

project_folder = "/content/drive/MyDrive/NLP_final_project"
os.makedirs(project_folder, exist_ok=True)

df_clean.to_parquet(
    f"{project_folder}/news_cleaned_notebook1.parquet",
    engine="pyarrow",
    index=False
)

print("saved to drive")

saved to drive
